# Volet A · Fine-tuning de Legal-BERT sur LEDGAR (jalon m4)

**Ce que fait ce notebook**
1. Fine-tuning **complet** de Legal-BERT (Trainer Hugging Face)
2. Fine-tuning **LoRA** avec r = 4, 8, 16
3. **Tableau de compromis** : paramètres entraînables, temps, mémoire GPU, accuracy, macro-F1
4. **Tests appariés** sur le même test set : McNemar (accuracy) et bootstrap apparié (macro-F1)

**Règles**
- Tous les choix (meilleure epoch, meilleur rang LoRA) se font sur la **validation**.
- Le test sert uniquement à rapporter les scores des modèles et à les comparer.
- Les résultats sont enregistrés dans Google Drive (`MyDrive/ledgar_A/`). Si Colab se déconnecte, relance toutes les cellules : les runs déjà terminés ne sont pas refaits.
- La section 11 exporte prédictions et métriques au format commun de l'équipe (`ledgar.metrics`). Les tableaux finaux, comparés à la **baseline commune**, sont produits par `python -m ledgar.finetune_report` (CPU, quelques secondes).



## 1 · Installation et données

GPU T4 pour l'entraînement. Les données sont les splits officiels de LEDGAR (LexGLUE), identiques à `ledgar.data.load_ledgar()`.

In [ ]:
import os, sys, torch
# Entraînement : GPU obligatoire (Exécution > Modifier le type d'exécution > GPU T4).
# Sections 9-10 seules : le CPU suffit.
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun (CPU)")

!pip install -q -U peft
!pip uninstall -y -q torchao   # la version de Colab est incompatible avec peft (LoRA)

# Même jeu de données que ledgar.data.load_ledgar() (splits officiels LexGLUE), sans dépendre du repo sur Colab
from datasets import load_dataset
def load_ledgar():
    ds = load_dataset("coastalcph/lex_glue", "ledgar")
    return ds, ds["train"].features["label"].names
def team_set_seed(seed=42):
    pass

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/ledgar_A"
os.makedirs(OUT, exist_ok=True)

## 2 · Configuration

**Pourquoi Legal-BERT *uncased* et pas CamemBERT ?** L'EDA a montré des textes parfois en majuscules (« T HIS A MENDMENT ») : un modèle *uncased* y est insensible. LEDGAR est en **anglais**, alors que CamemBERT est un modèle **français**. Legal-BERT a la même taille que BERT-base (~110 M de paramètres), mais il a été pré-entraîné sur des textes juridiques anglais, dont des contrats de la base **EDGAR de la SEC**, d'où viennent justement les clauses de LEDGAR.

In [ ]:
MODEL_NAME      = "nlpaueb/legal-bert-base-uncased"   # repli possible : "bert-base-uncased"
MAX_LENGTH      = 256      # décision de l'EDA : 86 % des clauses entières, attention 4x moins chère qu'à 512
SEED            = 42
BATCH_SIZE      = 16
EVAL_BATCH_SIZE = 64
EPOCHS          = 3
LR_FULL         = 2e-5     # fine-tuning complet : petit learning rate (S06)
LR_LORA         = 5e-4     # LoRA : learning rate plus élevé, peu de paramètres à apprendre
LORA_RANKS      = [4, 8, 16]
WARMUP_RATIO    = 0.1      # transformers 5 : warmup_steps < 1 = proportion des steps (S07)
WEIGHT_DECAY    = 0.01

## 3 · Données et tokenisation

In [ ]:
import json, time, gc, shutil
import numpy as np, pandas as pd
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)
from sklearn.metrics import accuracy_score, f1_score

team_set_seed()
set_seed(SEED)
raw, LABELS = load_ledgar()          # même chargement que 01_eda
N_CLASSES = len(LABELS)
assert len(raw["train"]) == 60000, "FAST_RUN actif : on veut les splits complets"
print(raw)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=MAX_LENGTH)

ds = raw.map(tokenize, batched=True, remove_columns=["text"])
collator = DataCollatorWithPadding(tok)   # padding dynamique, batch par batch
Y_VAL, Y_TEST = np.array(raw["validation"]["label"]), np.array(raw["test"]["label"])

# Décision de l'EDA (01_eda) : ~2,8 % du test est déjà dans le train (doublons exacts).
# On rapporte aussi la macro-F1 du test SANS ces clauses, avec la même normalisation que l'EDA.
norm = lambda t: " ".join(t.lower().split())   # = re.sub(r"\s+", " ", ...).strip() de 01_eda
train_norm = set(map(norm, raw["train"]["text"]))
TEST_LEAK = np.array([norm(t) in train_norm for t in raw["test"]["text"]])
print(f"clauses du test déjà présentes dans le train : {TEST_LEAK.mean():.1%} ({TEST_LEAK.sum()})")

## 4 · Fonctions : métriques et lancement d'une expérience

In [ ]:
def scores(y_true, y_pred):
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro"))}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = logits[0] if isinstance(logits, tuple) else logits
    return scores(labels, np.argmax(logits, -1))

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def logits_of(trainer, dataset):
    p = trainer.predict(dataset).predictions
    return p[0] if isinstance(p, tuple) else p

def save_preds(path, y_true, probs):
    pd.DataFrame({"idx": np.arange(len(y_true)), "y_true": y_true,
                  "y_pred": probs.argmax(1), "confidence": probs.max(1)}).to_csv(path, index=False)

def run_experiment(name, lora_r=None, lr=LR_FULL, epochs=EPOCHS, smoke=False):
    """Entraîne un modèle (complet si lora_r=None, LoRA sinon), garde la meilleure epoch
    selon la macro-F1 de VALIDATION, puis enregistre métriques et prédictions dans Drive."""
    run_dir = f"{OUT}/{name}"
    if not smoke and os.path.exists(f"{run_dir}/metrics.json"):
        print(f"[{name}] déjà terminé : je recharge les résultats.")
        return json.load(open(f"{run_dir}/metrics.json"))

    set_seed(SEED)
    train_ds, val_ds = ds["train"], ds["validation"]
    if smoke:   # test rapide du pipeline : 2 000 clauses, 1 epoch
        train_ds = train_ds.shuffle(seed=SEED).select(range(2000))
        val_ds, epochs = val_ds.select(range(1000)), 1

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_CLASSES,
        id2label=dict(enumerate(LABELS)), label2id={n: i for i, n in enumerate(LABELS)})
    if lora_r:
        from peft import LoraConfig, TaskType, get_peft_model
        # SEQ_CLS : la tête de classification reste entièrement entraînée
        model = get_peft_model(model, LoraConfig(
            task_type=TaskType.SEQ_CLS, r=lora_r, lora_alpha=2 * lora_r,
            lora_dropout=0.1, target_modules=["query", "value"]))   # W_Q et W_V
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"[{name}] paramètres entraînables : {trainable:,} / {total:,} ({100 * trainable / total:.2f} %)")

    args = TrainingArguments(
        output_dir=f"/content/ckpt/{name}",          # checkpoints en local, pas dans Drive
        learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE,
        warmup_steps=WARMUP_RATIO, weight_decay=WEIGHT_DECAY, lr_scheduler_type="linear",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        fp16=True,   # précision mixte sur GPU
        logging_steps=200, report_to="none", seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      processing_class=tok, data_collator=collator, compute_metrics=compute_metrics)

    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_out = trainer.train()
    train_min = (time.time() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    if smoke:
        est = train_out.metrics["train_runtime"] / 2000 * len(ds["train"]) / 60
        print(f"\nSMOKE TEST OK · validation : {trainer.evaluate()}")
        print(f"Estimation pour 1 epoch complète : ~{est:.0f} min "
              f"(soit ~{est * EPOCHS:.0f} min pour {EPOCHS} epochs) · pic mémoire GPU {peak_gb:.1f} Go")
        del trainer, model; gc.collect(); torch.cuda.empty_cache()
        return None

    os.makedirs(run_dir, exist_ok=True)
    # Prédictions avec le meilleur checkpoint (choisi sur la validation)
    val_probs = softmax(logits_of(trainer, ds["validation"]))
    t1 = time.time()
    test_probs = softmax(logits_of(trainer, ds["test"]))
    test_infer_s = time.time() - t1

    save_preds(f"{run_dir}/preds_val.csv", Y_VAL, val_probs)
    save_preds(f"{run_dir}/preds_test.csv", Y_TEST, test_probs)
    np.save(f"{run_dir}/probs_test.npy", test_probs.astype(np.float16))   # pour la calibration (volet C)
    json.dump(trainer.state.log_history, open(f"{run_dir}/log_history.json", "w"), indent=2)
    trainer.save_model(f"{run_dir}/model")   # modèle complet ou adaptateur LoRA

    metrics = {
        "run": name, "model": MODEL_NAME, "lora_r": lora_r, "lr": lr, "epochs": epochs,
        "max_length": MAX_LENGTH, "trainable_params": trainable, "total_params": total,
        "trainable_pct": round(100 * trainable / total, 3),
        "train_time_min": round(train_min, 1), "peak_gpu_mem_gb": round(peak_gb, 2),
        "test_inference_s": round(test_infer_s, 1),
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "validation": scores(Y_VAL, val_probs.argmax(1)),
        "test": scores(Y_TEST, test_probs.argmax(1)),
        "test_dedup": scores(Y_TEST[~TEST_LEAK], test_probs.argmax(1)[~TEST_LEAK]),
    }
    json.dump(metrics, open(f"{run_dir}/metrics.json", "w"), indent=2)
    print(json.dumps(metrics, indent=2))

    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"/content/ckpt/{name}", ignore_errors=True)
    return metrics

## 5 · Smoke test


In [ ]:
run_experiment("smoke", smoke=True)

## 6 · Fine-tuning complet

⚠️ **L'hypothèse doit être commitée dans le README avant de lancer cette cellule** (jalon m4).

In [ ]:
full = run_experiment("full_ft", lora_r=None, lr=LR_FULL)

## 6 bis · Fine-tuning complet jusqu'à convergence (exploratoire)

**Ajouté après l'hypothèse** : avec 3 epochs, la macro-F1 de validation montait encore (72,9 → 77,0 → 79,2). Même modèle, même learning rate, mais jusqu'à 15 epochs avec arrêt anticipé sur la macro-F1 de validation (patience 2), comme dans LexGLUE. Le run s'est arrêté à l'epoch 12 ; la meilleure epoch est la 10e (macro-F1 de validation 82,5 %).

Lancé sur un second compte Colab à cause de la limite de GPU ; résultats dans `ledgar_A/full_ft_es15`. **Ne pas relancer** (≈ 2 h 30 de GPU).

In [ ]:
# Ré-entraînement de Legal-BERT, jusqu'à 15 epochs + arrêt anticipé (patience 2)
# Checkpoints dans le Drive à chaque epoch : reprise automatique si Colab coupe

import dataclasses, glob
from transformers import EarlyStoppingCallback

RUN = "full_ft_es15"   # dossier Drive des résultats ; NE PAS relancer (≈ 2 h 30 de GPU)
CKPT_DIR = f"{OUT}/ckpt_{RUN}"

_BaseTrainer = Trainer
class ResumableESTrainer(_BaseTrainer):
    def __init__(self, *args, **kwargs):
        kwargs["args"] = dataclasses.replace(kwargs["args"], output_dir=CKPT_DIR)
        super().__init__(*args, **kwargs)
        self.add_callback(EarlyStoppingCallback(early_stopping_patience=2))
    def train(self, *args, **kwargs):
        last = sorted(glob.glob(f"{CKPT_DIR}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
        if last:
            print("Reprise depuis", last[-1])
            kwargs["resume_from_checkpoint"] = last[-1]
        return super().train(*args, **kwargs)

Trainer = ResumableESTrainer
try:
    full_es = run_experiment(RUN, lora_r=None, lr=LR_FULL, epochs=15)
finally:
    Trainer = _BaseTrainer

hist = json.load(open(f"{OUT}/{RUN}/log_history.json"))
curve = [(round(h["epoch"]), round(100 * h["eval_macro_f1"], 2)) for h in hist if "eval_macro_f1" in h]
print("macro-F1 de validation par epoch :", curve)

## 7 · LoRA : r = 4, 8, 16

In [ ]:
lora = {r: run_experiment(f"lora_r{r}", lora_r=r, lr=LR_LORA) for r in LORA_RANKS}

## 7 bis · LoRA r = 8 jusqu'à convergence (interrompu)

Même protocole que la section 6 bis. Le run a été coupé par la limite de GPU de Colab à l'epoch 10,6 sur 15, avant d'écrire ses checkpoints dans le Drive : **pas de résultats de test**. La macro-F1 de validation était de 78,4 % à l'epoch 10 (contre 82,5 % pour le fine-tuning complet au même stade). La comparaison LoRA / fine-tuning complet se fait donc à 3 epochs, avec le même protocole pour les deux.

In [ ]:
# LoRA r=8 jusqu'à convergence : même protocole que full_ft_es15
# (jusqu'à 15 epochs, arrêt anticipé sur la macro-F1 de validation, patience 2)

import dataclasses, glob
from transformers import EarlyStoppingCallback

RUN = "lora_r8_es10"
CKPT_DIR = f"{OUT}/ckpt_{RUN}"

_BaseTrainer = Trainer
class ResumableESTrainer(_BaseTrainer):
    def __init__(self, *args, **kwargs):
        kwargs["args"] = dataclasses.replace(kwargs["args"], output_dir=CKPT_DIR)
        super().__init__(*args, **kwargs)
        self.add_callback(EarlyStoppingCallback(early_stopping_patience=2))
    def train(self, *args, **kwargs):
        last = sorted(glob.glob(f"{CKPT_DIR}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
        if last:
            print("Reprise depuis", last[-1])
            kwargs["resume_from_checkpoint"] = last[-1]
        return super().train(*args, **kwargs)

Trainer = ResumableESTrainer
try:
    lora_es = run_experiment(RUN, lora_r=8, lr=LR_LORA, epochs=15)
finally:
    Trainer = _BaseTrainer

hist = json.load(open(f"{OUT}/{RUN}/log_history.json"))
curve = [(round(h["epoch"]), round(100 * h["eval_macro_f1"], 2)) for h in hist if "eval_macro_f1" in h]
print("macro-F1 de validation par epoch :", curve)

## 8 · Baseline TF-IDF locale (vérifications en cours de projet)

Baseline de travail (C = 10), utile pour les comparaisons dans ce notebook. **Les résultats finaux utilisent la baseline commune de l'équipe** (C = 30, `results/predictions/baseline_test.csv`), via `python -m ledgar.finetune_report`.

In [ ]:
def run_baseline(C=10.0, class_weight="balanced"):
    run_dir = f"{OUT}/baseline"
    if os.path.exists(f"{run_dir}/preds_test.csv"):
        print("Baseline déjà disponible."); return
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    os.makedirs(run_dir, exist_ok=True)
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)
    Xtr = vec.fit_transform(raw["train"]["text"])       # fit sur le train uniquement
    t0 = time.time()
    clf = LogisticRegression(C=C, class_weight=class_weight, max_iter=2000).fit(Xtr, raw["train"]["label"])
    train_min = (time.time() - t0) / 60
    val_probs = clf.predict_proba(vec.transform(raw["validation"]["text"]))
    test_probs = clf.predict_proba(vec.transform(raw["test"]["text"]))
    save_preds(f"{run_dir}/preds_val.csv", Y_VAL, val_probs)
    save_preds(f"{run_dir}/preds_test.csv", Y_TEST, test_probs)
    json.dump({"run": "baseline", "model": f"TF-IDF + LogReg (C={C}, {class_weight})",
               "lora_r": None, "trainable_params": int(clf.coef_.size + clf.intercept_.size),
               "total_params": int(clf.coef_.size + clf.intercept_.size), "trainable_pct": 100.0,
               "train_time_min": round(train_min, 1), "peak_gpu_mem_gb": 0.0,
               "validation": scores(Y_VAL, val_probs.argmax(1)),
               "test": scores(Y_TEST, test_probs.argmax(1)),
               "test_dedup": scores(Y_TEST[~TEST_LEAK], test_probs.argmax(1)[~TEST_LEAK])},
              open(f"{run_dir}/metrics.json", "w"), indent=2)
    print(json.load(open(f"{run_dir}/metrics.json")))

run_baseline()

## 9 · Tableau de compromis (version de travail)

Contre la baseline locale. Version finale : `results/tables/finetune_tradeoff.csv`.

In [ ]:
RUNS = ["baseline", "full_ft", "full_ft_es15"] + [f"lora_r{r}" for r in LORA_RANKS]
rows = []
for name in RUNS:
    p = f"{OUT}/{name}/metrics.json"
    if not os.path.exists(p):
        continue
    m = json.load(open(p))
    rows.append({
        "Modèle": name,
        "Params entraînables": f"{m['trainable_params']:,}",
        "% entraînés": m["trainable_pct"],
        "Temps (min)": m["train_time_min"],
        "Mémoire GPU (Go)": m["peak_gpu_mem_gb"],
        "Val macro-F1": round(100 * m["validation"]["macro_f1"], 2),
        "Test accuracy": round(100 * m["test"]["accuracy"], 2),
        "Test macro-F1": round(100 * m["test"]["macro_f1"], 2),
        "Test macro-F1 sans doublons": round(100 * m["test_dedup"]["macro_f1"], 2),
    })
table = pd.DataFrame(rows)
table.to_csv(f"{OUT}/tradeoff_table.csv", index=False)
display(table)
try:
    print(table.to_markdown(index=False))   # à coller dans le README
except ImportError:
    pass

## 10 · Tests appariés (même test set)

- **McNemar** : compare les accuracies en ne regardant que les clauses où un seul des deux modèles a raison.
- **Bootstrap apparié** : on rééchantillonne 1 000 fois le test set (le même tirage pour les deux modèles) et on regarde si l'intervalle à 95 % de la différence de macro-F1 contient 0.

Le meilleur LoRA est choisi sur la **validation**, pas sur le test.

In [ ]:
from scipy.stats import binomtest, chi2

def mcnemar(y, pa, pb):
    ok_a, ok_b = pa == y, pb == y
    b, c = int(np.sum(ok_a & ~ok_b)), int(np.sum(~ok_a & ok_b))
    if b + c < 25:
        p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
        return {"b_only_A_right": b, "c_only_B_right": c, "test": "exact binomial", "p_value": float(p)}
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    return {"b_only_A_right": b, "c_only_B_right": c, "test": "chi2 (continuity corr.)",
            "chi2": float(stat), "p_value": float(chi2.sf(stat, df=1))}

def paired_bootstrap_f1(y, pa, pb, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    n, diffs = len(y), np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)          # même tirage pour A et B
        diffs[i] = (f1_score(y[idx], pa[idx], average="macro")
                    - f1_score(y[idx], pb[idx], average="macro"))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"delta_macro_f1_pts": round(100 * (f1_score(y, pa, average="macro")
                                               - f1_score(y, pb, average="macro")), 2),
            "ci95_pts": [round(100 * lo, 2), round(100 * hi, 2)],
            "significant": bool(lo > 0 or hi < 0)}

def test_preds(name):
    return pd.read_csv(f"{OUT}/{name}/preds_test.csv")["y_pred"].values

done = [n for n in RUNS if os.path.exists(f"{OUT}/{n}/preds_test.csv")]
loras = [n for n in done if n.startswith("lora")]
best_lora = max(loras, key=lambda n: json.load(open(f"{OUT}/{n}/metrics.json"))["validation"]["macro_f1"]) if loras else None
print("Meilleur LoRA (choisi sur la validation) :", best_lora)

pairs = [("full_ft", "baseline"), ("full_ft", best_lora), (best_lora, "baseline"),
         ("full_ft_es15", "baseline"), ("full_ft_es15", "full_ft")]
stats = {}
for a, b in pairs:
    if a in done and b in done:
        pa, pb = test_preds(a), test_preds(b)
        stats[f"{a} vs {b}"] = {"mcnemar": mcnemar(Y_TEST, pa, pb),
                                "bootstrap": paired_bootstrap_f1(Y_TEST, pa, pb)}
        print(f"\n{a} vs {b}\n", json.dumps(stats[f'{a} vs {b}'], indent=2))
json.dump(stats, open(f"{OUT}/paired_tests.json", "w"), indent=2)

In [ ]:
# Où BERT perd-il ? F1 par groupe de fréquence (mêmes tiers que 01_eda)
counts = np.bincount(raw["train"]["label"], minlength=N_CLASSES)
rank = np.argsort(np.argsort(-counts))
group = np.where(rank < 34, "1-fréquente", np.where(rank < 67, "2-moyenne", "3-rare"))
models = ["baseline", "full_ft", "full_ft_es15", "lora_r8"]
f1s = {m: f1_score(Y_TEST, pd.read_csv(f"{OUT}/{m}/preds_test.csv").y_pred.values,
                   average=None, labels=range(N_CLASSES), zero_division=0) for m in models}
by_group = pd.DataFrame({m: pd.Series(f1s[m]).groupby(group).mean() * 100 for m in models}).round(1)
display(by_group)
by_group.to_csv(f"{OUT}/f1_by_freq_group.csv")

diff = pd.DataFrame({"classe": LABELS, "n_train": counts,
                     "F1 baseline": f1s["baseline"] * 100, "F1 BERT": f1s["full_ft"] * 100})
diff["écart"] = diff["F1 BERT"] - diff["F1 baseline"]
display(diff.sort_values("écart").head(10).round(1))
diff.to_csv(f"{OUT}/f1_per_class_baseline_vs_bert.csv", index=False)

## 11 · Export vers le repo de l'équipe

Convertit prédictions et métriques au format commun (`idx, label, pred, confidence`), à copier dans `results/predictions/` et `results/tables/` du repo. Les modèles (`model/`, `probs_test.npy`) restent dans le Drive partagé : trop lourds pour Git.

In [ ]:
# Export au format commun de l'équipe (ledgar.metrics) : à copier dans results/ du repo
# predictions/<modele>_{validation,test}.csv : idx, label, pred, confidence
# tables/finetune_metrics_<modele>.json et finetune_log_history_legalbert_full_es.json
EXPORT = "/content/export_volet_A"
NAMES = {"full_ft": "legalbert_full_3ep", "full_ft_es15": "legalbert_full_es",
         "lora_r4": "legalbert_lora_r4", "lora_r8": "legalbert_lora_r8", "lora_r16": "legalbert_lora_r16"}
os.makedirs(f"{EXPORT}/predictions", exist_ok=True); os.makedirs(f"{EXPORT}/tables", exist_ok=True)
for run, name in NAMES.items():
    for split, file in [("validation", "preds_val.csv"), ("test", "preds_test.csv")]:
        d = pd.read_csv(f"{OUT}/{run}/{file}")
        pd.DataFrame({"idx": d.idx, "label": d.y_true, "pred": d.y_pred,
                      "confidence": d.confidence.astype(float).round(6)}).to_csv(
            f"{EXPORT}/predictions/{name}_{split}.csv", index=False)
    m = json.load(open(f"{OUT}/{run}/metrics.json"))
    m["run"] = name
    m.pop("best_checkpoint", None)
    json.dump(m, open(f"{EXPORT}/tables/finetune_metrics_{name}.json", "w"), indent=2, ensure_ascii=False)
shutil.copy(f"{OUT}/full_ft_es15/log_history.json", f"{EXPORT}/tables/finetune_log_history_legalbert_full_es.json")
print(sorted(os.listdir(f"{EXPORT}/predictions")), sorted(os.listdir(f"{EXPORT}/tables")))
# Puis, dans le repo : python -m ledgar.finetune_report  -> tableaux finaux contre la baseline commune